In [1]:
import sys
import torch
import torch.nn.functional as F
from tqdm import tqdm
sys.path.append("..")
sys.path.append("../src")
from scripts.generate_soft_stabilities import load_model_dataset_attributions
from stability import soft_stability_rate

device = "cuda"

torch.manual_seed(1234)

/home/antonxue/lib/miniconda3/envs/tfl/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/antonxue/lib/miniconda3/envs/tfl/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/antonxue/lib/miniconda3/envs/tfl/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/

In [2]:
model, dataset, _ = load_model_dataset_attributions("../scripts/_cache", "resnet18", "imagenet_2_per_class", "random", 0.25)
model.eval().to(device);

In [3]:
num_samples = 100

In [4]:
# Find num_samples that correctly/incorrectly classify
images = []
good_attrs = []
bad_attrs = []
pbar = tqdm(range(len(dataset)))
for i in pbar:
    image = dataset[i].to(device)
    label = model(image[None,...]).view(-1).argmax().item()
    
    # Try to find a good_attr
    good_attr_found = False
    for _ in range(1000):
        good_attr = torch.zeros(196, device=device).long()
        good_attr[torch.randperm(196).to(device)[:49]] = 1
        if model(image[None,...], good_attr[None,...]).view(-1).argmax().item() == label:
            good_attr_found = True
            break

    # Then find the bad_attrs
    bad_attr_found = False
    for _ in range(1000):
        bad_attr = torch.zeros(196, device=device).long()
        bad_attr[torch.randperm(196, device=device)[:49]] = 1
        if model(image[None,...], bad_attr[None,...]).view(-1).argmax().item() != label:
            bad_attr_found = True
            break

    if good_attr_found and bad_attr_found:
        images.append(image)
        good_attrs.append(good_attr)
        bad_attrs.append(bad_attr)

    pbar.set_description(f"Found: {len(images)}")

    if len(images) >= num_samples:
        break

Found: 100:   9%|██████                                                            | 184/2000 [03:58<39:17,  1.30s/it]


In [5]:
radii = range(0, 21)
def compute_soft_stability_image(model, image, attr, radii=radii):
    soft_stability_rates = []
    for radius in radii:
        rate = soft_stability_rate(model, image, attr, radius, epsilon=0.1, delta=0.1)
        soft_stability_rates.append(round(rate.item(), 4))
    return torch.tensor(soft_stability_rates)

In [6]:
# Compute the stability rate of all the good ones
good_rates = []
for image, attr in tqdm(zip(images, good_attrs), total=len(images)):
    good_rates.append(compute_soft_stability_image(model, image, attr))
good_rates = torch.stack(good_rates)

100%|███████████████████████████████████████████████████████████████████████████████| 100/100 [03:22<00:00,  2.03s/it]


In [7]:
bad_rates = []
for image, attr in tqdm(zip(images, bad_attrs), total=len(images)):
    bad_rates.append(compute_soft_stability_image(model, image, attr))
bad_rates = torch.stack(bad_rates)

100%|███████████████████████████████████████████████████████████████████████████████| 100/100 [03:22<00:00,  2.02s/it]


In [8]:
good_rates

tensor([[1.0000, 0.5467, 0.4333,  ..., 0.4200, 0.5000, 0.4800],
        [1.0000, 0.8800, 0.7333,  ..., 0.4533, 0.3867, 0.4400],
        [1.0000, 0.9933, 0.9400,  ..., 0.9533, 0.9733, 0.9733],
        ...,
        [1.0000, 0.6467, 0.6200,  ..., 0.4000, 0.4333, 0.3867],
        [1.0000, 0.6400, 0.6067,  ..., 0.7467, 0.8200, 0.8133],
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000]])

In [9]:
good_rates.mean(dim=0)

tensor([1.0000, 0.8155, 0.7585, 0.7083, 0.6809, 0.6663, 0.6458, 0.6266, 0.6150,
        0.6045, 0.5940, 0.5948, 0.5941, 0.5889, 0.5885, 0.5804, 0.5851, 0.5779,
        0.5768, 0.5809, 0.5836])

In [10]:
bad_rates

tensor([[1.0000, 0.9400, 0.9067,  ..., 0.5600, 0.4267, 0.4933],
        [1.0000, 1.0000, 0.9867,  ..., 0.4067, 0.3067, 0.3800],
        [1.0000, 0.7733, 0.6733,  ..., 0.3267, 0.3133, 0.2600],
        ...,
        [1.0000, 0.5667, 0.4733,  ..., 0.0467, 0.0200, 0.0533],
        [1.0000, 0.4333, 0.3333,  ..., 0.0200, 0.0267, 0.0067],
        [1.0000, 0.9867, 0.9600,  ..., 0.1133, 0.0667, 0.0667]])

In [11]:
print("| Radius | Correctly Classified Mask | Misclassified Mask |")
print("| ------ | ------------------------- | ------------------ |")
for r in radii:
    print(f"| {r} | {good_rates.mean(dim=0)[r]:.3f} | {bad_rates.mean(dim=0)[r]:.3f}|")

| Radius | Correctly Classified Mask | Misclassified Mask |
| ------ | ------------------------- | ------------------ |
| 0 | 1.000 | 1.000|
| 1 | 0.815 | 0.856|
| 2 | 0.758 | 0.791|
| 3 | 0.708 | 0.727|
| 4 | 0.681 | 0.678|
| 5 | 0.666 | 0.636|
| 6 | 0.646 | 0.597|
| 7 | 0.627 | 0.570|
| 8 | 0.615 | 0.535|
| 9 | 0.605 | 0.510|
| 10 | 0.594 | 0.485|
| 11 | 0.595 | 0.464|
| 12 | 0.594 | 0.439|
| 13 | 0.589 | 0.427|
| 14 | 0.588 | 0.401|
| 15 | 0.580 | 0.382|
| 16 | 0.585 | 0.373|
| 17 | 0.578 | 0.355|
| 18 | 0.577 | 0.348|
| 19 | 0.581 | 0.329|
| 20 | 0.584 | 0.316|
